In [ ]:
!nvidia-smi

In [ ]:

%pip install -q kaggle pandas scikit-learn pillow matplotlib tqdm
%pip install -q torch torchvision
print("packages OK")

In [ ]:
# ===== Imports & hyperparameters =====
# NOTE: This is the TRAINING notebook.
#       All figures/metrics are produced by the separate analysis notebook.
import os, math, random, glob, json, shutil, warnings

os.environ["TORCH_HOME"] = "./torch_hub"
os.makedirs("./torch_hub", exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ----- Hyperparameters (tuned for a single RTX A5000, 24 GB) -----
SEED            = 42
MODEL_NAME      = "dinov2_vitb14_reg"    # 768-dim ViT-B/14 (+registers)
IMG_SIZE        = 224                    # 14 * 16; 14 is the size of one patch
BATCH_SIZE      = 64                     
EPOCHS          = 12
UNFREEZE_BLOCKS = 2                      # fine-tune only the last 2 backbone blocks 
LR_BACKBONE     = 1e-5
LR_HEAD         = 1e-3
WEIGHT_DECAY    = 0.05
NUM_HEADS       = 8
MLP_HIDDEN      = 512
DROPOUT         = 0.3
NUM_WORKERS     = 4                      
SUBSET_FRACTION = 1.0                    
WARMUP_FRAC     = 0.1
N_SAMPLE_IMAGES = 64                     # validation images copied for feature-map visualizing

DATA_DIR  = "./data"        
SAVE_DIR  = "./outputs"     
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_PATH = os.path.join(SAVE_DIR, "best_age_dinov2.pt")

device  = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = (device == "cuda")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True
print("device:", device, "| AMP:", use_amp)
print("working dir:", os.getcwd())

CONFIG = dict(model=MODEL_NAME, img_size=IMG_SIZE, batch=BATCH_SIZE, epochs=EPOCHS,
              unfreeze=UNFREEZE_BLOCKS, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
              num_heads=NUM_HEADS, mlp_hidden=MLP_HIDDEN, dropout=DROPOUT)

In [ ]:

import stat
kpath = "./kaggle.json"
assert os.path.exists(kpath), (
    "Missing ./kaggle.json . Download kaggle.json from your Kaggle account "
    "and place it in the same folder as this notebook."
)
try:
    os.chmod(kpath, stat.S_IRUSR | stat.S_IWUSR)
except Exception:
    pass
with open(kpath) as _f:
    _creds = json.load(_f)
os.environ["KAGGLE_USERNAME"] = _creds["username"]
os.environ["KAGGLE_KEY"]      = _creds["key"]
print("Kaggle credentials loaded from", kpath)

In [ ]:
import zipfile

ZIP_PATH = os.path.join(DATA_DIR, "face-age-gender-dataset.zip")
already_extracted = bool(glob.glob(os.path.join(DATA_DIR, "**", "train.csv"), recursive=True))

if not already_extracted:
    if not os.path.exists(ZIP_PATH):
        !kaggle datasets download -d aadyasingh55/face-age-gender-dataset -p "{DATA_DIR}"
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
print("Dataset ready under:", os.path.abspath(DATA_DIR))

In [ ]:

cands = glob.glob(os.path.join(DATA_DIR, "**", "train.csv"), recursive=True)
assert cands, "train.csv not found. Check the download/extraction."
TRAIN_CSV = cands[0]
BASE_DIR  = os.path.dirname(TRAIN_CSV)
print("train.csv:", TRAIN_CSV)

df = pd.read_csv(TRAIN_CSV)
print("columns:", list(df.columns), "| rows:", len(df))

# Index every image file: basename -> absolute path 
exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
all_imgs = []
for e in exts:
    all_imgs += glob.glob(os.path.join(DATA_DIR, "**", e), recursive=True)
print("total image files:", len(all_imgs))
basename_to_path = {os.path.basename(p): p for p in all_imgs}

def resolve_path(full_path):
    s = str(full_path)
    p = os.path.join(BASE_DIR, s)
    if os.path.exists(p):
        return p
    p2 = os.path.join(DATA_DIR, s)
    if os.path.exists(p2):
        return p2
    return basename_to_path.get(os.path.basename(s), None)

df["img_path"] = df["full_path"].map(resolve_path)
print("unresolved rows:", int(df["img_path"].isna().sum()))
df = df.dropna(subset=["img_path"]).reset_index(drop=True)


df["age"] = pd.to_numeric(df["age"], errors="coerce")
df = df.dropna(subset=["age"])
df = df[(df["age"] >= 1) & (df["age"] <= 100)].reset_index(drop=True)
print("final valid samples:", len(df))
print(df["age"].describe())

In [ ]:
# deviding train / val data
from sklearn.model_selection import train_test_split

work_df = df.sample(frac=SUBSET_FRACTION, random_state=SEED).reset_index(drop=True)
work_df["age_bin"] = pd.cut(work_df["age"],
                            bins=[0, 10, 18, 25, 35, 45, 55, 65, 200], labels=False)
try:
    train_df, val_df = train_test_split(
        work_df, test_size=0.2, random_state=SEED, stratify=work_df["age_bin"])
except ValueError:
    train_df, val_df = train_test_split(work_df, test_size=0.2, random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print("train=%d, val=%d" % (len(train_df), len(val_df)))

In [ ]:

AGE_MEAN = float(train_df["age"].mean())
AGE_STD  = float(train_df["age"].std())
print("AGE_MEAN=%.2f, AGE_STD=%.2f" % (AGE_MEAN, AGE_STD))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
RESIZE = int(round(IMG_SIZE * 1.15))

train_tf = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FaceAgeDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["img_path"].values
        self.ages  = frame["age"].values.astype("float32")
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        img = self.transform(img)
        y = (self.ages[i] - AGE_MEAN) / AGE_STD   # normalized target
        return img, torch.tensor(y, dtype=torch.float32)

train_ds = FaceAgeDataset(train_df, train_tf)
val_ds   = FaceAgeDataset(val_df,   eval_tf)

persistent = NUM_WORKERS > 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                          persistent_workers=persistent)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=persistent)
print("batches: train=%d, val=%d" % (len(train_loader), len(val_loader)))

In [ ]:

class AttentionPooling(nn.Module):
    # A learnable query aggregates patch tokens via cross-attention into a dim-vector
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attn  = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm  = nn.LayerNorm(dim)
    def forward(self, tokens):                 # tokens: [B, N, dim]
        B = tokens.size(0)
        q = self.query.expand(B, -1, -1)
        pooled, _ = self.attn(q, tokens, tokens)
        return self.norm(pooled.squeeze(1))    # [B, dim]

class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden=512, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x)

class AgeRegressor(nn.Module):
    def __init__(self, backbone, embed_dim, num_heads, hidden, dropout):
        super().__init__()
        self.backbone = backbone
        self.pool = AttentionPooling(embed_dim, num_heads)
        self.head = MLPHead(embed_dim, hidden, dropout)
    def forward(self, x):
        feats = self.backbone.forward_features(x)
        patch_tokens = feats["x_norm_patchtokens"]   # [B, N, 768]
        pooled = self.pool(patch_tokens)             # [B, 768]
        return self.head(pooled).squeeze(-1)         # [B]

backbone = torch.hub.load("facebookresearch/dinov2", MODEL_NAME, trust_repo=True)
EMBED_DIM = backbone.embed_dim
print("DINOv2 embed_dim =", EMBED_DIM, "| patch_size =", backbone.patch_size)
assert EMBED_DIM == 768, "This pipeline assumes a 768-dim (ViT-B/14) backbone."

model = AgeRegressor(backbone, EMBED_DIM, NUM_HEADS, MLP_HIDDEN, DROPOUT).to(device)
print("total parameters: %.1fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

In [ ]:
# Fine-tuning scope: train only the last 2 backbone blocks =====
def get_transformer_blocks(bb):
    blocks = []
    for item in bb.blocks:
        if hasattr(item, "attn"):
            blocks.append(item)
        else:                      # container such as BlockChunk
            for sub in item.children():
                if hasattr(sub, "attn"):
                    blocks.append(sub)
    return blocks

for p in model.backbone.parameters():
    p.requires_grad = False

blocks = get_transformer_blocks(model.backbone)
print("backbone transformer blocks:", len(blocks))
if UNFREEZE_BLOCKS > 0:
    for blk in blocks[-UNFREEZE_BLOCKS:]:
        for p in blk.parameters():
            p.requires_grad = True
    if getattr(model.backbone, "norm", None) is not None:
        for p in model.backbone.norm.parameters():
            p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print("trainable parameters: %.1fM" % trainable)

In [ ]:
# Loss / optimizer / scheduler
backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
head_params     = list(model.pool.parameters()) + list(model.head.parameters())

param_groups = [{"params": head_params, "lr": LR_HEAD}]
if backbone_params:
    param_groups.append({"params": backbone_params, "lr": LR_BACKBONE})

optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
criterion = nn.SmoothL1Loss(beta=1.0)      # Huber loss in the normalized age space

total_steps  = EPOCHS * max(1, len(train_loader))
warmup_steps = int(WARMUP_FRAC * total_steps)
def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / max(1, warmup_steps)
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = GradScaler("cuda", enabled=use_amp)
print("ready. total_steps=%d, warmup_steps=%d" % (total_steps, warmup_steps))

In [ ]:
# Run one epoch (train or eval)
def run_epoch(loader, training):
    model.train(training)
    total_loss, n = 0.0, 0
    preds_all, gts_all = [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, ys in tqdm(loader, leave=False):
            imgs = imgs.to(device, non_blocking=True)
            ys   = ys.to(device, non_blocking=True)
            with autocast("cuda", enabled=use_amp):
                out  = model(imgs)
                loss = criterion(out, ys)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            bs = imgs.size(0)
            total_loss += loss.item() * bs; n += bs
            preds_all.append(out.detach().float().cpu())
            gts_all.append(ys.detach().float().cpu())
    preds = torch.cat(preds_all) * AGE_STD + AGE_MEAN     # convert back to years
    gts   = torch.cat(gts_all)   * AGE_STD + AGE_MEAN
    mae   = (preds - gts).abs().mean().item()
    return total_loss / max(1, n), mae

In [ ]:
# Training loop
# Save timing: best checkpoint (model + age_mean/std + config) to SAVE_DIR whenever val MAE improves.
# Save: best_age_dinov2.pt

best_mae = float("inf")
history = {"train_loss": [], "train_mae": [], "val_loss": [], "val_mae": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_mae = run_epoch(train_loader, True)
    va_loss, va_mae = run_epoch(val_loader, False)
    history["train_loss"].append(tr_loss); history["train_mae"].append(tr_mae)
    history["val_loss"].append(va_loss);   history["val_mae"].append(va_mae)
    print("[%02d/%d] train: loss=%.4f MAE=%.2fyr | val: loss=%.4f MAE=%.2fyr"
          % (epoch, EPOCHS, tr_loss, tr_mae, va_loss, va_mae))
    if va_mae < best_mae:
        best_mae = va_mae
        torch.save({"model": model.state_dict(), "age_mean": AGE_MEAN,
                    "age_std": AGE_STD, "config": CONFIG}, CKPT_PATH)
        print("    best updated (val MAE=%.2fyr) -> saved: %s" % (best_mae, CKPT_PATH))

print("\nBest validation MAE = %.2f years" % best_mae)

In [ ]:
# Save all artifacts for the analysis notebook 


# This cell writes everything the results-analysis notebook needs:
# best_age_dinov2.pt: trained weights + age mean/std + config + history
# (the checkpoint from the loop is re-saved here with `history` added)

# history.json: per-epoch train/val loss & MAE  (for the loss curves)

# val_predictions.csv: full_path, true_age, pred_age for every validation image
# (used for actual-vs-predicted plots and all metrics)

# all_ages.csv: the full cleaned age column     
# (for the age-distribution plot)

# sample_images/ + samples_manifest.csv: random validation images + their true ages
# (used for the feature-map visualizations)

# Reload the BEST checkpoint 
# because current weights may be from the last epoch, not the best one
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"]); model.eval()

# Predict on the whole validation set 
# val_loader is shuffle=False, so order == val_df

preds = []
with torch.no_grad():
    for imgs, ys in tqdm(val_loader, desc="val inference"):
        with autocast("cuda", enabled=use_amp):
            out = model(imgs.to(device))
        preds.append(out.float().cpu())
pred_years = (torch.cat(preds) * AGE_STD + AGE_MEAN).numpy()

# checkpoint (+history)
ckpt["history"] = history
torch.save(ckpt, CKPT_PATH)
print("SAVED:", CKPT_PATH)

# history.json
with open(os.path.join(SAVE_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)
print("SAVED:", os.path.join(SAVE_DIR, "history.json"))

# val_predictions.csv
pd.DataFrame({
    "full_path": val_df["full_path"].values,
    "true_age":  val_df["age"].values.astype(float),
    "pred_age":  pred_years,
}).to_csv(os.path.join(SAVE_DIR, "val_predictions.csv"), index=False)
print("SAVED:", os.path.join(SAVE_DIR, "val_predictions.csv"))

# all_ages.csv
df[["age"]].to_csv(os.path.join(SAVE_DIR, "all_ages.csv"), index=False)
print("SAVED:", os.path.join(SAVE_DIR, "all_ages.csv"))

# random validation images + manifest
sample_dir = os.path.join(SAVE_DIR, "sample_images")
os.makedirs(sample_dir, exist_ok=True)
samp = val_df.sample(n=min(N_SAMPLE_IMAGES, len(val_df)), random_state=SEED)
rows = []
for _, r in samp.iterrows():
    fn = os.path.basename(str(r["img_path"]))
    try:
        shutil.copy(r["img_path"], os.path.join(sample_dir, fn))
        rows.append({"filename": fn, "true_age": int(r["age"])})
    except Exception:
        pass
pd.DataFrame(rows).to_csv(os.path.join(SAVE_DIR, "samples_manifest.csv"), index=False)
print("SAVED: %d sample images -> %s" % (len(rows), sample_dir))

print("\nAll artifacts saved to:", os.path.abspath(SAVE_DIR))
print("Best validation MAE = %.2f years" % best_mae)